# Aftershock: checks on a finished panel

One cell, no GPU and no MODELLER needed. It reads a panel that is already on your Google Drive and

1. checks whether the metal site stayed coordinated in every simulation (`pert site-check`);
2. re-derives the null-control verdicts with the current rules, reusing the WT-vs-WT pairs already computed;
3. downloads the tables, figures and reports (without trajectories) as a zip.

Run the cell below and accept the Google Drive request. Reading the trajectories from Drive takes a few minutes.
If it fails, copy the text from the line `CHECKS FAILED` to the end.


In [ ]:
PANEL = "pdz3_pilot"                                    # folder of the panel inside RUN_ROOT
RUN_ROOT = "/content/drive/MyDrive/aftershock_runs"     # where the main notebook wrote it
REPO = "juvenalyosa/aftershock"
CODE = "/content/aftershock"

import importlib.util, json, os, subprocess, sys, traceback


def step(title):
    print(f"\n=== {title} ===", flush=True)


def shell(*command, **kwargs):
    done = subprocess.run(list(command), capture_output=True, text=True, **kwargs)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command[:4])} ... failed:\n{(done.stdout + done.stderr)[-3000:]}")
    return done.stdout


try:
    step("1/5 Google Drive")
    from google.colab import drive, files
    drive.mount("/content/drive")
    PANEL_DIR = f"{RUN_ROOT}/{PANEL}"
    if not os.path.isdir(PANEL_DIR):
        found = sorted(os.listdir(RUN_ROOT)) if os.path.isdir(RUN_ROOT) else "that folder does not exist"
        raise RuntimeError(f"No panel at {PANEL_DIR}. In {RUN_ROOT}: {found}")
    print("Panel:", PANEL_DIR)

    step("2/5 Code")
    if os.path.isdir(f"{CODE}/.git"):
        shell("git", "-C", CODE, "pull", "--ff-only")
    else:
        shell("git", "clone", f"https://github.com/{REPO}.git", CODE)
    print("Code:", shell("git", "-C", CODE, "log", "-1", "--format=%h %s").strip())
    if importlib.util.find_spec("openmm") is None:      # only its structure reader is needed
        shell(sys.executable, "-m", "pip", "install", "-q", "openmm")
    sys.path.insert(0, f"{CODE}/scripts")
    import colab_tools as ct
    import pandas as pd
    from IPython.display import Image, display

    step("3/5 Is the metal site intact? (reads every trajectory)")
    recorded = json.load(open(f"{PANEL_DIR}/null_control/null_control_summary.json")).get("ligand_resnames") or ["ZN"]
    code = ct.run_stream(
        [sys.executable, "-m", "pert", "site-check", "--panel-dir", PANEL_DIR, "--ligand-resname", ",".join(recorded)],
        cwd=CODE, env={"MPLBACKEND": "Agg"},
    )
    if code != 0:
        raise RuntimeError("pert site-check failed; its message is in the lines above.")
    site = json.load(open(f"{PANEL_DIR}/site_check/site_check_summary.json"))
    print("Simulations by status:", site["status_counts_by_system"])
    display(pd.read_csv(f"{PANEL_DIR}/site_check/site_check_simulations.csv"))
    display(Image(f"{PANEL_DIR}/site_check/site_check.png"))

    step("4/5 Verdicts under the current rules")
    shell(sys.executable, "-m", "pert", "null-control", "--panel-dir", PANEL_DIR, "--recalibrate",
          cwd=CODE, env={**os.environ, "MPLBACKEND": "Agg"})
    for note in json.load(open(f"{PANEL_DIR}/null_control/null_control_summary.json"))["notes"]:
        print(" -", note)
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(ct.verdict_rows(PANEL_DIR)))

    step("5/5 Download")
    files.download(ct.zip_light_results(PANEL_DIR, f"/content/{PANEL}_results.zip"))
    print("\nCHECKS FINISHED")
except Exception:
    print("\nCHECKS FAILED\n")
    traceback.print_exc()
